## Feature Engineering
In diesem Abschnitt transformieren wir die bereinigten Rohdaten in aussagekräftige Prädiktoren (Features), die dem Machine-Learning-Modell helfen, die sportliche Dynamik und das Kräfteverhältnis im Peloton besser zu verstehen. Wir konzentrieren uns auf drei Schlüsselbereiche:

* **Fahrer-Physis (BMI):** * **Was wir tun:** Wir berechnen den Body-Mass-Index ($BMI$) aus der Körpergröße (`height` in Metern) und dem Gewicht (`weight` in kg).
    * **Warum:** Die reine Masse oder Größe reicht nicht aus. Das Verhältnis (BMI) ist im Radsport der Indikator für den Fahrertyp: Ein niedriger BMI deutet auf einen leichten Bergfahrer (Kletterspezialist) hin, ein höherer BMI auf einen kraftvollen Sprinter oder Zeitfahrer.

* **Qualität des Fahrerfeldes (Race Competitiveness):**
    * **Was wir tun:** Wir aggregieren die Weltranglisten-Positionen (`rider_rank_season`) aller Fahrer, die an einer spezifischen Etappe teilnehmen.
    * **Warum:** Ein Rennen ist nicht gleich ein Rennen. Durch die Summe (oder den Median) der Ranglistenplätze der Teilnehmer pro Etappe weiß das Modell sofort, ob es sich um eine hochkarätig besetzte Tour-de-France-Königsetappe oder ein schwächer besetztes Vorbereitungsrennen handelt.
    * Da 9999- Werte in der Weltrangliste vorhanden nehmen wir den Median

* **Team-Stärke (Team Power Index):**
    * **Was wir tun:** Wir berechnen für jede Etappe die kumulierte Stärke eines Teams, indem wir die Weltranglisten-Positionen aller Fahrer *desselben* Teams auf dieser spezifischen Etappe aggregieren.
    * **Warum:** Radsport ist ein Teamsport. Ein Top-Favorit mit einer extrem starken Team (starke Helfer mit niedrigen Ranglisten-Plätzen) im Rücken hat statistisch eine völlig andere Siegchance als ein Einzelkämpfer in einem schwachen Team.

In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_pickle('../../data/processed/19_cleaned_master_data.pkl')

In [3]:
# BMI berechnen
df['rider_bmi'] = df['weight'] / (df['height'] ** 2)

# Rennstärke berechnen

etappen_gruppen = df.groupby(['race', 'year', 'stage_nr'])
df['race_competitiveness_median'] = etappen_gruppen['rider_rank_season'].transform('median')


# Teamstärken bestimmen
etappen_team_gruppen = df.groupby(['race', 'year', 'stage_nr', 'current_team'])

df['team_power_index'] = etappen_team_gruppen['rider_rank_season'].transform('median')



# Kontrolle
features_kontrolle = ['name', 'current_team', 'rider_bmi', 'race_competitiveness_median', 'team_power_index']
print(df[features_kontrolle].drop_duplicates().head(5).to_string(index=False))

                    name                     current_team  rider_bmi  race_competitiveness_median  team_power_index
              Tom Boonen           Quickstep - Innergetic  22.243924                        219.0             345.0
            Thor Hushovd                  Crédit Agricole  24.784258                        219.0             119.0
           Robbie McEwen                Davitamon - Lotto  22.913033                        219.0             195.0
          Stuart O'Grady Cofidis, le Crédit par Téléphone  23.566632                        219.0             206.0
Luciano André Pagliarini                         Liquigas  22.460034                        219.0             223.0


## Checkpoint       

In [4]:
pfad = '../../data/processed/20_cleaned_master_data.pkl'
df.to_pickle(pfad)